# NB11b — external cohort ds004584: fixed GCN grid and controls, shard 0 (CPU)

The unchanged 7 x 3 protocol as a two-class (PD vs control) problem on the frozen ds004584 folds; shard 0 adds the non-graph twins, shard 1 the leakage experiment and 100 label permutations for the configuration selected on ds004504 (P3 x spatial).

In [ ]:
REPO = 'https://github.com/Devguru-codes/Major_project_IIITN.git'
BRANCH = 'feat/implementation'
WORK = '/kaggle/working'
SRC = '/tmp/eegrep-src'   # code checkout stays out of the saved output

import subprocess, sys, os, json, platform, shutil

def sh(cmd, log=None):
    print('$', cmd, flush=True)
    r = subprocess.run(cmd, shell=True, text=True, capture_output=True)
    print(r.stdout[-20000:], r.stderr[-5000:], sep='\n', flush=True)
    if log:
        open(log, 'w').write(r.stdout + '\n' + r.stderr)
    if r.returncode != 0:
        raise RuntimeError(f'command failed ({r.returncode}): {cmd}')
    return r.stdout

shutil.rmtree(SRC, ignore_errors=True)
sh(f'git clone --depth 1 -b {BRANCH} {REPO} {SRC}')
SHA = sh(f'git -C {SRC} rev-parse --short HEAD').strip()
sh(f'pip install -q -e "{SRC}[dev]"', log=f'{WORK}/pip_install.log')
PY = f'cd {SRC} && {sys.executable} -m eegrep'
print('git sha', SHA)

In [ ]:
sh(f'cd {SRC} && {sys.executable} -m pytest -q -p no:cacheprovider tests/test_external.py tests/test_model_train.py', log=f'{WORK}/pytest_ext.txt')

In [ ]:
import glob
CACHE = [d for d in glob.glob('/kaggle/input/**/cache', recursive=True) if os.path.exists(f'{d}/index.npz')][0]
PART = sorted(glob.glob('/kaggle/input/**/participants.tsv', recursive=True))[0]
FOLDS = f'{SRC}/splits/folds_ds004504.json'
print(CACHE, PART, FOLDS)
EXT = f'{PY} --config configs/ds004584.yaml'
assert os.path.exists(f'{SRC}/splits/folds_ds004584.json'), 'commit the NB11a folds first'

In [ ]:
sh(f'{EXT} grid --cache {CACHE} --folds {SRC}/splits/folds_ds004584.json --pipelines P1,P2,P3,P4 --tag main --out {WORK}/ext_runs.jsonl --max-hours 6', log=f'{WORK}/grid_0.log')

In [ ]:
sh(f'{EXT} classical --cache {CACHE} --folds {SRC}/splits/folds_ds004584.json --fit-on train --out {WORK}/cls_train.csv', log=f'{WORK}/classical_train.log')
sh(f'{EXT} classical --cache {CACHE} --folds {SRC}/splits/folds_ds004584.json --fit-on trainval --out {WORK}/cls_tv.csv', log=f'{WORK}/classical_tv.log')
import pandas as pd
pd.concat([pd.read_csv(f'{WORK}/cls_train.csv'), pd.read_csv(f'{WORK}/cls_tv.csv')]).to_csv(f'{WORK}/classical.csv', index=False)

In [ ]:
import torch
sh(f'{sys.executable} -m pip freeze', log=f'{WORK}/pip_freeze.txt')
env = {'git_sha': SHA, 'python': platform.python_version(), 'torch': torch.__version__,
       'cuda': torch.cuda.is_available(), 'cpu_count': os.cpu_count(),
       'kaggle_image': os.environ.get('KAGGLE_DOCKER_IMAGE', 'unknown')}
json.dump(env, open(f'{WORK}/environment.json', 'w'), indent=1)
print(env)